# Project: Market Research System

Companion notebook for the CrewAI learning platform: [Agent Foundry](https://agent-foundry-pi.vercel.app)


In [ ]:
!pip install -q crewai crewai-tools pydantic

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")
serper = getpass("Enter your Serper API key (optional; needed for SerperDevTool): ")
if serper.strip():
    os.environ["SERPER_API_KEY"] = serper


## Step 1: Define the Pydantic output model

`MarketReport` shapes the final deliverable: **industry**, **trends**, **competitors**, **opportunities**, and **summary**.


In [ ]:
from pydantic import BaseModel


class MarketReport(BaseModel):
    industry: str
    trends: list[str]
    competitors: list[str]
    opportunities: list[str]
    summary: str


## Step 2: Define the agents

- **Market Researcher** — `SerperDevTool` + `ScrapeWebsiteTool`
- **Data Analyst** — interprets research
- **Report Writer** — fills the structured report


In [ ]:
from crewai import Agent
from crewai_tools import ScrapeWebsiteTool, SerperDevTool

search_tool = SerperDevTool()
scrape_tool = ScrapeWebsiteTool()

market_researcher = Agent(
    role="Market Researcher",
    goal="Find accurate, current market information using search and reputable sources",
    backstory="You search the web, open relevant pages when needed, and collect facts without inventing data.",
    tools=[search_tool, scrape_tool],
    verbose=True,
)

data_analyst = Agent(
    role="Data Analyst",
    goal="Turn raw market research into clear trends, competitor notes, and strategic implications",
    backstory="You are rigorous about separating evidence from speculation and you structure findings for decision-makers.",
    verbose=True,
)

report_writer = Agent(
    role="Report Writer",
    goal="Deliver a concise, structured market report aligned with the agreed schema",
    backstory="You write executive-ready summaries and ensure every section maps to the required fields.",
    verbose=True,
)


## Step 3: Define the tasks

- **research_task** — gather data for `{industry}`
- **analysis_task** — `context=[research_task]`
- **report_task** — `output_pydantic=MarketReport`, context from research + analysis


In [ ]:
from crewai import Task

research_task = Task(
    description=(
        "Research the market for the industry: {industry}. "
        "Collect key facts, notable companies, recent developments, and source-backed notes."
    ),
    expected_output="Organized research notes with themes, data points, and URLs or source hints where available.",
    agent=market_researcher,
)

analysis_task = Task(
    description=(
        "Using the research, identify major trends, main competitors or segments, and plausible opportunities or risks "
        "for {industry}. Be explicit about what is supported by the research versus inference."
    ),
    expected_output="A structured analysis: trends, competitors, opportunities/risks, and a short narrative synthesis.",
    agent=data_analyst,
    context=[research_task],
)

report_task = Task(
    description=(
        "Produce the final market report for {industry}. "
        "Fill industry, trends, competitors, opportunities, and summary to match the schema exactly."
    ),
    expected_output="A MarketReport instance with all required fields populated from the prior research and analysis.",
    agent=report_writer,
    context=[research_task, analysis_task],
    output_pydantic=MarketReport,
)


## Step 4: Assemble the crew

Use **`Process.sequential`** so research runs before analysis, and analysis before the structured report.


In [ ]:
from crewai import Crew, Process

crew = Crew(
    agents=[market_researcher, data_analyst, report_writer],
    tasks=[research_task, analysis_task, report_task],
    process=Process.sequential,
    verbose=True,
)


## Step 5: Run with kickoff

Pass **`industry`** in `inputs` (example: **AI Agents**).


In [ ]:
result = crew.kickoff(inputs={"industry": "AI Agents"})
print(result)
if result.pydantic:
    print(result.pydantic.model_dump_json(indent=2))


## What you learned

You built a **sequential crew** with **search and scrape tools**, used **task context** to pass outputs forward, and ended with **`output_pydantic`** so the final artifact is a typed **`MarketReport`**. Together, that is the pattern for research-style pipelines that must return reliable structured data.
